# Test — Animation : chute libre (avec et sans frottements)

Notebook de test (jetable) pour évaluer `matplotlib.animation.FuncAnimation` exportée en
`to_jshtml()` comme alternative à `ipywidgets.interact` pour les animations continues.

**Objectif** : vérifier que l'animation (lecteur JS autonome, sans noyau requis) s'affiche
correctement une fois le site MyST construit, contrairement aux cellules `interact()` qui
restent bloquées sur `Loading...` dans le site statique.

On compare deux solutions de chute libre à partir d'une même hauteur $z_0$, sans vitesse
initiale :

- **Sans frottements** : $\ddot{z} = -g$
- **Avec frottements fluides** (linéaires, $\overrightarrow{f} = -k\overrightarrow{v}$) :
  $\ddot{z} = -g - \dfrac{k}{m}\dot{z}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## Solutions analytiques

**Sans frottements** (chute libre classique) :

$$
z(t) = z_0 - \frac{1}{2}g t^2 \qquad v(t) = -g t
$$

**Avec frottements fluides linéaires** : en posant $\tau = m/k$ (constante de temps) et
$v_{\text{lim}} = -mg/k$ (vitesse limite, négative car dirigée vers le bas) :

$$
v(t) = v_{\text{lim}} \left(1 - e^{-t/\tau}\right)
\qquad
z(t) = z_0 + v_{\text{lim}} t + v_{\text{lim}} \tau \left(e^{-t/\tau} - 1\right)
$$

In [ ]:
# Paramètres physiques
g = 9.81      # m/s^2
z0 = 20.0     # m, hauteur initiale
m = 1.0       # kg
k = 1.0       # kg/s, coefficient de frottement fluide

tau = m / k
v_lim = -m * g / k


def z_sans(t):
    return z0 - 0.5 * g * t**2


def z_avec(t):
    return z0 + v_lim * t + v_lim * tau * (np.exp(-t / tau) - 1.0)


# Temps où la solution "sans frottements" touche le sol (z = 0)
t_max = np.sqrt(2 * z0 / g)
print(f"tau = {tau:.2f} s, v_lim = {v_lim:.2f} m/s, t_max (sans frottements) = {t_max:.2f} s")

In [ ]:
n_frames = 60
temps = np.linspace(0, t_max, n_frames)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 3)
ax.set_ylim(0, z0 + 1)
ax.set_xlabel("(horizontal — écarté seulement pour la lisibilité)")
ax.set_ylabel(r"$z$ [m]")
ax.set_title("Chute libre : avec vs sans frottements")
ax.axhline(0, color="black", linewidth=1)

(point_sans,) = ax.plot([0], [z0], "o", color="tab:blue", markersize=14, label="Sans frottements")
(point_avec,) = ax.plot([2], [z0], "o", color="tab:orange", markersize=14, label="Avec frottements")
ax.legend(loc="upper right")
time_text = ax.text(0.05, 0.95, "", transform=ax.transAxes, va="top")


def init():
    point_sans.set_data([0], [z0])
    point_avec.set_data([2], [z0])
    time_text.set_text("")
    return point_sans, point_avec, time_text


def update(frame):
    t = temps[frame]
    point_sans.set_data([0], [max(z_sans(t), 0)])
    point_avec.set_data([2], [max(z_avec(t), 0)])
    time_text.set_text(f"t = {t:.2f} s")
    return point_sans, point_avec, time_text


anim = FuncAnimation(fig, update, frames=n_frames, init_func=init, interval=60, blit=True)
plt.close(fig)  # évite d'afficher aussi l'image statique du dernier frame
HTML(anim.to_jshtml())

::::{note} Interprétation
Le point **sans frottements** accélère indéfiniment ($v(t) = -gt$), tandis que le point
**avec frottements** tend vers sa vitesse limite $v_{\text{lim}}$ et touche donc le sol plus
tard.
::::

## Test : widget ipywidgets pour le coefficient de frottement

On teste ici si un slider `ipywidgets` (`interact`) est une bonne approche pour faire
varier le coefficient de frottement $k$ et **régénérer** l'animation matplotlib à chaque
changement.

::::{attention}
Ceci nécessite un noyau vif — comme toutes les cellules `interact()`, ça reste bloqué sur
`Loading...` dans le site statique MyST (déjà confirmé précédemment dans ce cours). Ici on
teste seulement si l'expérience **en local / noyau vivant** est utilisable, avant de
décider si ça vaut la peine de l'intégrer dans un vrai TP.
::::

In [ ]:
from ipywidgets import interact, FloatSlider
from IPython.display import display


def animer_chute(k_frottement=1.0):
    tau_local = m / k_frottement
    v_lim_local = -m * g / k_frottement

    def z_avec_local(t):
        return z0 + v_lim_local * t + v_lim_local * tau_local * (np.exp(-t / tau_local) - 1.0)

    fig, ax = plt.subplots(figsize=(4, 6))
    ax.set_xlim(-1, 3)
    ax.set_ylim(0, z0 + 1)
    ax.set_xlabel("(horizontal — écarté seulement pour la lisibilité)")
    ax.set_ylabel(r"$z$ [m]")
    ax.set_title(f"Chute libre — k = {k_frottement:.2f} kg/s")
    ax.axhline(0, color="black", linewidth=1)

    (point_sans,) = ax.plot([0], [z0], "o", color="tab:blue", markersize=14, label="Sans frottements")
    (point_avec,) = ax.plot([2], [z0], "o", color="tab:orange", markersize=14, label="Avec frottements")
    ax.legend(loc="upper right")
    time_text = ax.text(0.05, 0.95, "", transform=ax.transAxes, va="top")

    def init():
        point_sans.set_data([0], [z0])
        point_avec.set_data([2], [z0])
        time_text.set_text("")
        return point_sans, point_avec, time_text

    def update(frame):
        t = temps[frame]
        point_sans.set_data([0], [max(z_sans(t), 0)])
        point_avec.set_data([2], [max(z_avec_local(t), 0)])
        time_text.set_text(f"t = {t:.2f} s")
        return point_sans, point_avec, time_text

    anim_local = FuncAnimation(fig, update, frames=n_frames, init_func=init, interval=60, blit=True)
    plt.close(fig)
    display(HTML(anim_local.to_jshtml()))


interact(
    animer_chute,
    k_frottement=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, continuous_update=False),
);